# Task 076: CDI — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Coherent diffraction imaging phase retrieval using HIO and ER

**Paper**: See repository documentation
**Repository**: https://github.com/clatlan/cdiutils

### Physical Background

Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.

### Forward Model

```
y = H * x + n  (PSF convolution)
```

### Inverse Problem

```
x_hat = argmin 1/2 ||H*x - y||^2 + lambda R(x)
```

### Performance Metrics
- **PSNR**: 20.32 dB ← 🔧 修复前: 19.46 dB
- **SSIM**: 0.159
- **Evaluation**: Coherent diffraction imaging HIO+ER phase retrieval (CC=0.926)

### Mathematical Formulation

Coherent diffraction measures only intensity, losing phase information:

$$I(\mathbf{q}) = |\mathcal{F}\{\psi(\mathbf{r})\}|^2$$

where $\psi(\mathbf{r})$ is the complex wavefield and $\mathcal{F}$ is the Fourier transform.

**HIO Algorithm** (Hybrid Input-Output):
$$\psi'^{(k)} = \mathcal{P}_F(\psi^{(k)}) \quad \text{(Fourier constraint)}$$
$$\psi^{(k+1)}(\mathbf{r}) = \begin{cases} \psi'^{(k)}(\mathbf{r}) & \mathbf{r} \in S \\ \psi^{(k)}(\mathbf{r}) - \beta\,\psi'^{(k)}(\mathbf{r}) & \mathbf{r} \notin S \end{cases}$$

**Error Reduction**: $\psi^{(k+1)} = \mathcal{P}_S \circ \mathcal{P}_F(\psi^{(k)})$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
CDI — Coherent Diffraction Imaging Phase Retrieval
=====================================================
Task #73: Reconstruct complex-valued object from its far-field
          diffraction intensity pattern using iterative phase retrieval.

Inverse Problem:
    Given |F{ρ(r)}|² (diffraction intensity), recover ρ(r) (object).
    The phase information is lost — this is the "phase problem".

Forward Model:
    Far-field diffraction (Fraunhofer):
    I(q) = |F{ρ(r)}|² = |Σ_r ρ(r) exp(-i q·r)|²
    Only intensity is measured; phase is lost.

Inverse Solver:
    1) HIO (Hybrid Input-Output) — Fienup's algorithm
    2) ER (Error Reduction) — Gerchberg-Saxton
    3) HIO+ER alternating with shrink-wrap support update

Repo: https://github.com/mcherukara/CDI_image_reconstruction
Paper: Fienup (1982), Applied Optics; Marchesini (2003), Rev. Sci. Instr.

Usage: /data/yjh/spectro_env/bin/python CDI_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`hio_iteration`, `er_iteration`, `update_support_shrinkwrap`, `phase_retrieval`, `align_phase`

In [ ]:
# ─── Inverse Solver: HIO ──────────────────────────────────────────
def hio_iteration(obj_est, sqrt_intensity, support, beta):
    """
    Single HIO (Hybrid Input-Output) iteration.

    1. FFT → replace amplitude with measured √I → IFFT
    2. Apply support constraint with HIO feedback
    """
    # Fourier constraint: replace amplitude
    F_est = fftshift(fft2(ifftshift(obj_est)))
    phase_est = np.angle(F_est)
    F_constrained = sqrt_intensity * np.exp(1j * phase_est)
    obj_new = fftshift(ifft2(ifftshift(F_constrained)))

    # Real-space constraint (HIO)
    obj_out = obj_est.copy()
    outside_support = ~support

    # Inside support: keep new estimate
    obj_out[support] = obj_new[support]
    # Outside support: HIO feedback
    obj_out[outside_support] = obj_est[outside_support] - beta * obj_new[outside_support]

    return obj_out

def er_iteration(obj_est, sqrt_intensity, support):
    """
    Single ER (Error Reduction) iteration.
    Same as HIO but with hard support projection.
    """
    F_est = fftshift(fft2(ifftshift(obj_est)))
    phase_est = np.angle(F_est)
    F_constrained = sqrt_intensity * np.exp(1j * phase_est)
    obj_new = fftshift(ifft2(ifftshift(F_constrained)))

    # Hard support constraint
    obj_new[~support] = 0
    return obj_new

def update_support_shrinkwrap(obj_est, sigma, threshold):
    """
    Shrink-wrap support estimation: blur amplitude, threshold.
    """
    amp = np.abs(obj_est)
    amp_smooth = gaussian_filter(amp, sigma=sigma)
    amp_smooth /= max(amp_smooth.max(), 1e-12)
    support = amp_smooth > threshold
    # Dilate slightly for stability
    support = binary_dilation(support, iterations=1)
    return support

def phase_retrieval(intensity_noisy, support_init, n_hio, n_er, beta,
                     shrinkwrap_interval, shrinkwrap_sigma, shrinkwrap_threshold,
                     rng):
    """
    Full HIO+ER phase retrieval with shrink-wrap support.
    """
    sqrt_intensity = np.sqrt(intensity_noisy)
    det_size = intensity_noisy.shape[0]

    # Random initial guess
    phase_init = 2 * np.pi * rng.random((det_size, det_size))
    obj_est = support_init.astype(complex) * np.exp(1j * phase_init)

    support = support_init.copy()
    errors = []

    print(f"  Phase retrieval: {n_hio} HIO + {n_er} ER iterations")

    # HIO phase
    for it in range(n_hio):
        obj_est = hio_iteration(obj_est, sqrt_intensity, support, beta)

        # Shrink-wrap support update
        if (it + 1) % shrinkwrap_interval == 0:
            support = update_support_shrinkwrap(
                obj_est, shrinkwrap_sigma, shrinkwrap_threshold
            )
            print(f"    HIO iter {it+1:4d}: support pixels = {support.sum()}")

        # Error metric
        F_est = fftshift(fft2(ifftshift(obj_est)))
        err = np.sqrt(np.mean((np.abs(F_est) - sqrt_intensity)**2))
        errors.append(err)

        if (it + 1) % 50 == 0:
            print(f"    HIO iter {it+1:4d}: R-factor = {err:.6f}")

    # ER phase (refinement)
    for it in range(n_er):
        obj_est = er_iteration(obj_est, sqrt_intensity, support)

        if (it + 1) % 25 == 0:
            F_est = fftshift(fft2(ifftshift(obj_est)))
            err = np.sqrt(np.mean((np.abs(F_est) - sqrt_intensity)**2))
            errors.append(err)
            print(f"    ER  iter {it+1:4d}: R-factor = {err:.6f}")

    return obj_est, support, errors

# ─── Phase Alignment ──────────────────────────────────────────────
def align_phase(obj_gt, obj_rec):
    """
    Remove global phase ambiguity and possible twin-image flip.
    """
    # Try direct and conjugate
    candidates = [obj_rec, np.conj(obj_rec), np.flip(obj_rec),
                  np.conj(np.flip(obj_rec))]

    best_cc = -1
    best = obj_rec

    for cand in candidates:
        # Find optimal global phase
        cross = np.sum(obj_gt * np.conj(cand))
        phi = np.angle(cross)
        cand_aligned = cand * np.exp(1j * phi)

        cc = np.abs(np.corrcoef(
            np.abs(obj_gt).ravel(), np.abs(cand_aligned).ravel()
        )[0, 1])
        if cc > best_cc:
            best_cc = cc
            best = cand_aligned

    return best

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
y = H * x + n  (PSF convolution)
```

Functions: `generate_object`, `forward_diffraction`

In [ ]:
# ─── Data Generation ──────────────────────────────────────────────
def generate_object(obj_size, det_size):
    """
    Create a complex-valued object with amplitude and phase structure.
    Represents a nanocrystal or biological sample.
    """
    # Amplitude: geometric features
    amp = np.zeros((det_size, det_size))
    cx, cy = det_size // 2, det_size // 2
    Y, X = np.mgrid[:det_size, :det_size]

    # Crystal-like shape
    r = np.sqrt((X - cx)**2 + (Y - cy)**2)
    amp[r < obj_size // 3] = 0.8

    # Internal structure
    amp[(np.abs(X - cx) < obj_size // 6) &
        (np.abs(Y - cy) < obj_size // 4)] = 1.0

    # Small features
    amp[(X - cx + 10)**2 + (Y - cy + 8)**2 < 16] = 0.6
    amp[(X - cx - 8)**2 + (Y - cy - 12)**2 < 9] = 0.7

    # Phase: smooth strain field
    phase = np.zeros((det_size, det_size))
    phase = 0.5 * np.sin(2 * np.pi * (X - cx) / obj_size) * (amp > 0)
    phase += 0.3 * np.cos(2 * np.pi * (Y - cy) / (obj_size * 0.8)) * (amp > 0)

    obj = amp * np.exp(1j * phase)
    support = amp > 0.01

    return obj, support

def forward_diffraction(obj, snr_db, rng):
    """
    Compute far-field diffraction intensity pattern.
    I = |FFT(obj)|²
    Add Poisson-like photon noise.
    """
    F_obj = fftshift(fft2(ifftshift(obj)))
    intensity_clean = np.abs(F_obj)**2

    # Normalise
    intensity_clean /= intensity_clean.max()

    # Poisson-like noise
    sig_power = np.mean(intensity_clean**2)
    noise_power = sig_power / (10**(snr_db / 10))
    noise = np.sqrt(noise_power) * rng.standard_normal(intensity_clean.shape)
    intensity_noisy = np.maximum(intensity_clean + noise, 0)

    return intensity_clean, intensity_noisy, F_obj

## 5. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize_results`

In [ ]:
# ─── Metrics ───────────────────────────────────────────────────────
def compute_metrics(obj_gt, obj_rec):
    """Compute reconstruction metrics on amplitude."""
    amp_gt = np.abs(obj_gt)
    amp_rec = np.abs(obj_rec)

    amp_gt_n = amp_gt / max(amp_gt.max(), 1e-12)
    amp_rec_n = amp_rec / max(amp_rec.max(), 1e-12)

    data_range = 1.0
    mse = np.mean((amp_gt_n - amp_rec_n)**2)
    psnr = float(10 * np.log10(data_range**2 / max(mse, 1e-30)))
    ssim_val = float(ssim_fn(amp_gt_n, amp_rec_n, data_range=data_range))
    cc = float(np.corrcoef(amp_gt_n.ravel(), amp_rec_n.ravel())[0, 1])
    re = float(np.linalg.norm(amp_gt_n - amp_rec_n) /
               max(np.linalg.norm(amp_gt_n), 1e-12))
    rmse = float(np.sqrt(mse))

    # Phase error (inside support only)
    support = amp_gt > 0.01 * amp_gt.max()
    if support.sum() > 0:
        phase_gt = np.angle(obj_gt[support])
        phase_rec = np.angle(obj_rec[support])
        phase_err = np.angle(np.exp(1j * (phase_gt - phase_rec)))
        phase_rmse = float(np.sqrt(np.mean(phase_err**2)))
    else:
        phase_rmse = np.pi

    return {
        "PSNR": psnr, "SSIM": ssim_val, "CC": cc, "RE": re, "RMSE": rmse,
        "phase_RMSE_rad": phase_rmse,
    }

# ─── Visualization ─────────────────────────────────────────────────
def visualize_results(obj_gt, obj_rec, intensity, errors, metrics, save_path):
    fig, axes = plt.subplots(2, 4, figsize=(22, 10))

    # Diffraction pattern
    axes[0, 0].imshow(np.log10(intensity + 1e-6), cmap='viridis')
    axes[0, 0].set_title('Diffraction Pattern (log)')

    # GT amplitude
    axes[0, 1].imshow(np.abs(obj_gt), cmap='gray')
    axes[0, 1].set_title('GT Amplitude')

    # Recon amplitude
    axes[0, 2].imshow(np.abs(obj_rec), cmap='gray')
    axes[0, 2].set_title('Recon Amplitude')

    # Amplitude error
    err_amp = np.abs(np.abs(obj_gt) - np.abs(obj_rec))
    axes[0, 3].imshow(err_amp, cmap='hot')
    axes[0, 3].set_title('|Amplitude Error|')

    # GT phase
    support = np.abs(obj_gt) > 0.01 * np.abs(obj_gt).max()
    phase_gt = np.angle(obj_gt) * support
    axes[1, 0].imshow(phase_gt, cmap='twilight', vmin=-np.pi, vmax=np.pi)
    axes[1, 0].set_title('GT Phase')

    # Recon phase
    phase_rec = np.angle(obj_rec) * support
    axes[1, 1].imshow(phase_rec, cmap='twilight', vmin=-np.pi, vmax=np.pi)
    axes[1, 1].set_title('Recon Phase')

    # Phase error
    phase_err = np.angle(np.exp(1j * (phase_gt - phase_rec))) * support
    axes[1, 2].imshow(phase_err, cmap='RdBu_r', vmin=-1, vmax=1)
    axes[1, 2].set_title('Phase Error')

    # Convergence
    if errors:
        axes[1, 3].semilogy(errors)
        axes[1, 3].set_title('Convergence (R-factor)')
        axes[1, 3].set_xlabel('Iteration')
        axes[1, 3].grid(True)

    for row in axes:
        for ax in row:
            ax.axis('off') if ax != axes[1, 3] else None

    fig.suptitle(
        f"CDI — Phase Retrieval (HIO+ER)\n"
        f"PSNR={metrics['PSNR']:.1f} dB | CC={metrics['CC']:.4f} | "
        f"Phase RMSE={metrics['phase_RMSE_rad']:.3f} rad",
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("=" * 70)
print("  CDI — Coherent Diffraction Imaging Phase Retrieval")
print("=" * 70)

rng = np.random.default_rng(SEED)

### Stage 1: Data Generation

Intermediate processing step in the pipeline.

In [ ]:
# Stage 1: Data Generation
print("\n[STAGE 1] Data Generation")
obj_gt, support_gt = generate_object(OBJ_SIZE, DET_SIZE)
print(f"  Object: {obj_gt.shape} (complex)")
print(f"  Support pixels: {support_gt.sum()}")
print(f"  Oversampling: {DET_SIZE/OBJ_SIZE:.1f}×")

### Stage 2: Forward — Diffraction

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Stage 2: Forward — Diffraction
print("\n[STAGE 2] Forward — Far-Field Diffraction I = |FFT(ρ)|²")
intensity_clean, intensity_noisy, F_gt = forward_diffraction(
    obj_gt, NOISE_SNR_DB, rng
)
print(f"  Intensity range: [{intensity_noisy.min():.4f}, "
      f"{intensity_noisy.max():.4f}]")

### Stage 3: Inverse — Phase Retrieval (multi-start)

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Stage 3: Inverse — Phase Retrieval (multi-start)
print(f"\n[STAGE 3] Inverse — HIO + ER Phase Retrieval ({N_STARTS} starts)")
best_obj_rec = None
best_err = float('inf')
best_errors = []
best_support = None
for start_i in range(N_STARTS):
    rng_start = np.random.default_rng(SEED + start_i)
    obj_rec_i, support_i, errors_i = phase_retrieval(
        intensity_noisy, support_gt, N_HIO, N_ER, BETA_HIO,
        SHRINKWRAP_INTERVAL, SHRINKWRAP_SIGMA, SHRINKWRAP_THRESHOLD, rng_start
    )
    final_err = errors_i[-1] if errors_i else float('inf')
    print(f"  Start {start_i+1}/{N_STARTS}: final R-factor = {final_err:.6f}")
    if final_err < best_err:
        best_err = final_err
        best_obj_rec = obj_rec_i
        best_errors = errors_i
        best_support = support_i
obj_rec = best_obj_rec
support_final = best_support
errors = best_errors
print(f"  Best start: R-factor = {best_err:.6f}")

# Align phase
obj_rec = align_phase(obj_gt, obj_rec)

### Stage 4: Evaluation

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Stage 4: Evaluation
print("\n[STAGE 4] Evaluation Metrics:")
metrics = compute_metrics(obj_gt, obj_rec)
for k, v in sorted(metrics.items()):
    print(f"  {k:20s} = {v}")

# Standard metrics
std_metrics = {k: v for k, v in metrics.items()
               if k in ["PSNR", "SSIM", "CC", "RE", "RMSE"]}
std_metrics["phase_RMSE_rad"] = metrics["phase_RMSE_rad"]

with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(std_metrics, f, indent=2)
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), np.abs(obj_rec))
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), np.abs(obj_gt))

visualize_results(obj_gt, obj_rec, intensity_noisy, errors, metrics,
                  os.path.join(RESULTS_DIR, "reconstruction_result.png"))

print("\n" + "=" * 70)
print("  DONE — Results saved to", RESULTS_DIR)
print("=" * 70)

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **CDI**:

1. **Problem Setup**: Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=20.32 dB ← 🔧 修复前: 19.46 dB, SSIM=0.159)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: See documentation
- Repository: https://github.com/clatlan/cdiutils
